# Deney — Kaş Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion

Bu notebook yalnızca mevcut **kaş ROI framelerini** kullanır. Yeniden yüz/kaş kırpma işlemi yapmaz.

**Girdi**
- `AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş/metadata.csv`
- Kaş görselleri: `Kaş/real/...` ve `Kaş/fake/...`
- Video kimliği eşleştirmesi: `Deneyler/Deney 1 Frame/secim_metadata.csv`

**Özellikler**
- LBP: 54
- GLCM: 72
- Gabor: 72
- Wavelet: 42
- Toplam handcrafted özellik: **240**

**Model**
- Swin V2-Tiny görüntü kolu
- 240 boyutlu texture özellik kolu
- Fusion classifier → `real / fake`

**Çıktı**
Her çalıştırmada sonuçlar şu dizine yazılır:

`AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/<run_id>/`

Klasör biçimi:
- `checkpoints/`
- `logs/`
- `metrics/`
- `predictions/`
- `figures/`
- `artifacts/`
- `config_resolved.yaml`
- `requirements_lock.txt`
- `environment.json`
- `output_manifest.csv`
- `run_summary.json`

In [1]:
# ============================================================
# 0. COLAB SETUP
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!pip -q install \
    "torch>=2.3" "torchvision>=0.18" \
    "scikit-image>=0.24" "PyWavelets>=1.6" \
    "scikit-learn>=1.4" "pandas>=2.0" "numpy>=1.26" \
    "matplotlib>=3.8" "Pillow>=10.0" "PyYAML>=6.0" \
    "tqdm>=4.66" "joblib>=1.3" "opencv-python-headless>=4.9"

print("Kurulum tamamlandı.")

Mounted at /content/drive
Kurulum tamamlandı.


In [2]:
# ============================================================
# 1. DRIVE PATH PREFLIGHT
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AISC DeepFake Çalışmaları"
)
EXPERIMENT_ROOT = (
    PROJECT_ROOT / "Deneyler" / "Nazlıcan" / "Deney 1"
)
EYEBROW_ROOT = EXPERIMENT_ROOT / "Kaş"
RESULTS_ROOT = EXPERIMENT_ROOT / "Sonuçlar"
SELECTION_METADATA = (
    PROJECT_ROOT / "Deneyler" / "Deney 1 Frame" / "secim_metadata.csv"
)
ROI_METADATA = EYEBROW_ROOT / "metadata.csv"

required_paths = [
    PROJECT_ROOT,
    EXPERIMENT_ROOT,
    EYEBROW_ROOT,
    RESULTS_ROOT,
    SELECTION_METADATA,
    ROI_METADATA,
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Aşağıdaki gerekli Drive yolları bulunamadı:\n"
        + "\n".join(missing)
    )

print("Kaş klasörü:", EYEBROW_ROOT)
print("Kaş metadata:", ROI_METADATA)
print("Seçim metadata:", SELECTION_METADATA)
print("Sonuç klasörü:", RESULTS_ROOT)
print("Drive yol kontrolü başarılı.")

Kaş klasörü: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş
Kaş metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş/metadata.csv
Seçim metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv
Sonuç klasörü: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar
Drive yol kontrolü başarılı.


In [3]:
# ============================================================
# 2. CONFIGURATION
# Bu blok deney ayarlarının tek kaynağıdır.
# ============================================================
CONFIG = {
    "seed": 42,
    "region": "eyebrow",
    "model_name": "swinv2_t_lbp_glcm_gabor_wavelet_fusion",

    # Drive paths
    "data_root": str(EYEBROW_ROOT),
    "roi_metadata_path": str(ROI_METADATA),
    "selection_metadata_path": str(SELECTION_METADATA),
    "results_root": str(RESULTS_ROOT),

    # ROI metadata columns
    "sample_id_column": "sample_id",
    "path_column": "output_path",
    "relative_path_column": "output_relative_path",
    "input_path_column": "input_path",
    "label_column": "label",
    "split_column": "split",
    "status_column": "status",
    "success_status_values": ["SUCCESS", "OK"],

    # Image and Swin V2
    "image_size": 224,
    "pretrained": True,
    "freeze_backbone_epochs": 5,
    "unfreeze_last_stages": True,

    # Handcrafted texture features
    "lbp_radii": [1, 2, 3],
    "lbp_points": [8, 16, 24],
    "glcm_distances": [1, 2, 4],
    "glcm_angles_deg": [0, 45, 90, 135],
    "glcm_levels": 32,
    "gabor_orientations_deg": [0, 30, 60, 90, 120, 150],
    "gabor_frequencies": [0.10, 0.20, 0.30],
    "wavelet": "db2",
    "wavelet_level": 2,

    # Training
    "batch_size": 16,
    "num_workers": 2,
    "epochs": 30,
    "learning_rate": 2e-4,
    "backbone_learning_rate": 2e-5,
    "weight_decay": 1e-4,
    "dropout": 0.30,
    "hidden_dim": 256,
    "patience": 8,
    "threshold": 0.50,
    "threshold_search_min": 0.10,
    "threshold_search_max": 0.90,
    "threshold_search_steps": 161,
    "amp": True,
    "gradient_clip_norm": 1.0,

    # Execution controls
    "smoke_test": True,
    "smoke_train_batches": 2,
    "smoke_val_batches": 2,
    "feature_cache_flush_every": 100,
    "overwrite_feature_cache": False,
    "save_feature_arrays": True,

    # Default: Swin + all 240 handcrafted features.
    "active_mode": "full",
    "run_all_ablations": False,
    "ablation_modes": [
        "swin_only",
        "texture_only",
        "swin_lbp",
        "swin_lbp_glcm",
        "swin_lbp_glcm_gabor",
        "full",
    ],

    # Kesilen bir çalışmayı aynı klasörden sürdürmek için run_id yazılabilir.
    "resume_run_id": None,
}

assert CONFIG["active_mode"] in CONFIG["ablation_modes"]
print("Configuration loaded.")

Configuration loaded.


In [4]:
# ============================================================
# 3. IMPORTS, REPRODUCIBILITY, RUN DIRECTORY AND ROOT ARTIFACTS
# ============================================================
import os
import gc
import io
import sys
import json
import yaml
import math
import time
import copy
import random
import hashlib
import logging
import platform
import subprocess
import warnings
import importlib.metadata as importlib_metadata

from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import cv2
import joblib
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt

from PIL import Image, ImageOps
from IPython.display import display
from tqdm.auto import tqdm

from skimage.feature import (
    local_binary_pattern,
    graycomatrix,
    graycoprops,
)
from skimage.filters import gabor

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import swin_v2_t, Swin_V2_T_Weights
from torchvision import transforms

warnings.filterwarnings("ignore", category=UserWarning)

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_STARTED_AT = datetime.now(timezone.utc)
RUN_START_MONOTONIC = time.monotonic()

if CONFIG["resume_run_id"]:
    RUN_ID = str(CONFIG["resume_run_id"])
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    RUN_ID = (
        f"{timestamp}_eyebrow_swinv2_lbp_glcm_"
        f"gabor_wavelet_seed{CONFIG['seed']}"
    )

RUN_DIR = Path(CONFIG["results_root"]) / RUN_ID
DIRS = {
    "run": RUN_DIR,
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

# Log file
LOGGER = logging.getLogger("eyebrow_experiment")
LOGGER.handlers.clear()
LOGGER.setLevel(logging.INFO)
file_handler = logging.FileHandler(
    DIRS["logs"] / "pipeline.log",
    encoding="utf-8",
)
stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
LOGGER.addHandler(file_handler)
LOGGER.addHandler(stream_handler)

def log(message: str) -> None:
    LOGGER.info(message)

# Resolve and save the exact configuration.
CONFIG_RESOLVED = dict(CONFIG)
CONFIG_RESOLVED.update({
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "device": str(DEVICE),
    "created_at_utc": RUN_STARTED_AT.isoformat(),
})
with open(
    RUN_DIR / "config_resolved.yaml",
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        CONFIG_RESOLVED,
        file,
        allow_unicode=True,
        sort_keys=False,
    )

# Environment metadata.
package_names = [
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "scikit-learn",
    "scikit-image",
    "PyWavelets",
    "Pillow",
    "opencv-python-headless",
    "joblib",
]
package_versions = {}
for name in package_names:
    try:
        package_versions[name] = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        package_versions[name] = None

environment_payload = {
    "python": sys.version,
    "platform": platform.platform(),
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "packages": package_versions,
}
with open(
    RUN_DIR / "environment.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

# Exact pip environment.
try:
    requirements_text = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    )
except Exception as exc:
    requirements_text = f"pip freeze failed: {type(exc).__name__}: {exc}\n"

with open(
    RUN_DIR / "requirements_lock.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write(requirements_text)

log(f"Device: {DEVICE}")
log(f"Run ID: {RUN_ID}")
log(f"Run directory: {RUN_DIR}")

2026-08-06 21:24:37,969 | INFO | Device: cuda


INFO:eyebrow_experiment:Device: cuda


2026-08-06 21:24:37,972 | INFO | Run ID: 20260806_2124_eyebrow_swinv2_lbp_glcm_gabor_wavelet_seed42


INFO:eyebrow_experiment:Run ID: 20260806_2124_eyebrow_swinv2_lbp_glcm_gabor_wavelet_seed42


2026-08-06 21:24:37,975 | INFO | Run directory: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260806_2124_eyebrow_swinv2_lbp_glcm_gabor_wavelet_seed42


INFO:eyebrow_experiment:Run directory: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260806_2124_eyebrow_swinv2_lbp_glcm_gabor_wavelet_seed42


In [5]:
# ============================================================
# 4. ATOMIC I/O HELPERS
# ============================================================
def atomic_write_bytes(data: bytes, target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    with open(temp, "wb") as file:
        file.write(data)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temp, target)

def atomic_write_text(text: str, target: Path) -> None:
    atomic_write_bytes(text.encode("utf-8"), target)

def atomic_write_json(payload: Dict[str, Any], target: Path) -> None:
    atomic_write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        target,
    )

def atomic_write_csv(dataframe: pd.DataFrame, target: Path) -> None:
    buffer = io.StringIO()
    dataframe.to_csv(buffer, index=False)
    atomic_write_text(buffer.getvalue(), target)

def atomic_joblib_dump(obj: Any, target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    joblib.dump(obj, temp)
    _ = joblib.load(temp)
    os.replace(temp, target)

def atomic_torch_save(state: Dict[str, Any], target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    torch.save(state, temp)
    loaded = torch.load(
        temp,
        map_location="cpu",
        weights_only=False,
    )
    required = {
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
    }
    if not required.issubset(loaded):
        raise RuntimeError(
            f"Checkpoint validation failed: {temp}"
        )
    os.replace(temp, target)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("Atomic I/O helpers ready.")

Atomic I/O helpers ready.


In [6]:
# ============================================================
# 5. LOAD KAŞ METADATA + RECOVER SOURCE VIDEO IDS
# ============================================================
roi_metadata_path = Path(CONFIG["roi_metadata_path"])
selection_metadata_path = Path(CONFIG["selection_metadata_path"])
data_root = Path(CONFIG["data_root"])

roi = pd.read_csv(roi_metadata_path)
selection = pd.read_csv(
    selection_metadata_path,
    encoding="utf-8-sig",
)

required_roi_columns = {
    CONFIG["sample_id_column"],
    CONFIG["path_column"],
    CONFIG["relative_path_column"],
    CONFIG["input_path_column"],
    CONFIG["label_column"],
    CONFIG["split_column"],
    CONFIG["status_column"],
}
missing_roi_columns = sorted(required_roi_columns - set(roi.columns))
if missing_roi_columns:
    raise KeyError(
        f"Kaş metadata dosyasında eksik sütunlar: {missing_roi_columns}"
    )

required_selection_columns = {
    "sinif",
    "split",
    "orijinal_yol",
    "yeni_yol",
    "dosya_adi",
}
missing_selection_columns = sorted(
    required_selection_columns - set(selection.columns)
)
if missing_selection_columns:
    raise KeyError(
        "Seçim metadata dosyasında eksik sütunlar: "
        f"{missing_selection_columns}"
    )

# Normalize ROI metadata.
roi[CONFIG["status_column"]] = (
    roi[CONFIG["status_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)
roi[CONFIG["label_column"]] = (
    roi[CONFIG["label_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
roi[CONFIG["split_column"]] = (
    roi[CONFIG["split_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "training": "train",
        "validation": "val",
        "valid": "val",
        "testing": "test",
    })
)

# Normalize selection metadata.
selection["selection_label"] = (
    selection["sinif"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
selection["selection_split"] = (
    selection["split"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
selection["frame_name"] = (
    selection["dosya_adi"]
    .fillna("")
    .astype(str)
    .map(lambda value: Path(value).name)
)
selection["source_video_path"] = (
    selection["orijinal_yol"]
    .fillna("")
    .astype(str)
)
selection["source_video_name"] = selection[
    "source_video_path"
].map(
    lambda value: (
        Path(value).parent.name
        if value
        else ""
    )
)

if selection["frame_name"].duplicated().any():
    duplicate_names = (
        selection.loc[
            selection["frame_name"].duplicated(keep=False),
            "frame_name",
        ]
        .drop_duplicates()
        .head(20)
        .tolist()
    )
    raise ValueError(
        "secim_metadata.csv içinde tekrarlanan dosya adları var: "
        f"{duplicate_names}"
    )

# Resolve each eyebrow image.
def resolve_output_path(row: pd.Series) -> str:
    absolute_value = str(
        row.get(CONFIG["path_column"], "")
    ).strip()
    relative_value = str(
        row.get(CONFIG["relative_path_column"], "")
    ).strip()

    if absolute_value:
        absolute_path = Path(absolute_value)
        if absolute_path.is_absolute():
            return str(absolute_path)

    if relative_value:
        return str(data_root / relative_value)

    return absolute_value

roi["resolved_image_path"] = roi.apply(
    resolve_output_path,
    axis=1,
)
roi["frame_name"] = roi[
    CONFIG["input_path_column"]
].fillna("").astype(str).map(
    lambda value: Path(value).name
)

total_roi_rows = len(roi)
successful_roi = roi[
    roi[CONFIG["status_column"]].isin(
        CONFIG["success_status_values"]
    )
    & roi[CONFIG["label_column"]].isin(["real", "fake"])
    & roi[CONFIG["split_column"]].isin(["train", "val", "test"])
].copy()

successful_roi["file_exists"] = successful_roi[
    "resolved_image_path"
].map(lambda value: Path(value).exists())

missing_success_files = successful_roi[
    ~successful_roi["file_exists"]
].copy()
successful_roi = successful_roi[
    successful_roi["file_exists"]
].copy()

# Merge ROI rows with the original frame-selection metadata.
selection_for_merge = selection[[
    "frame_name",
    "selection_label",
    "selection_split",
    "source_video_path",
    "source_video_name",
]].copy()

valid = successful_roi.merge(
    selection_for_merge,
    on="frame_name",
    how="left",
    validate="many_to_one",
    indicator=True,
)

# Merge başarısını veri içeriğinden değil, pandas merge göstergesinden kontrol et.
unmatched = valid[valid["_merge"] != "both"].copy()
if len(unmatched) > 0:
    atomic_write_csv(
        unmatched,
        DIRS["metrics"] / "metadata_unmatched_rows.csv",
    )
    raise RuntimeError(
        f"{len(unmatched)} başarılı kaş frame'i seçim metadata ile "
        "eşleşmedi. Ayrıntı metrics/metadata_unmatched_rows.csv içinde."
    )
valid = valid.drop(columns=["_merge"])

# Eşleşmiş satırda orijinal video yolu boşsa bu bir eşleşme hatası değil,
# kaynak metadata kalite hatasıdır; ayrı raporla ve açık mesaj ver.
missing_source_video = valid[
    valid["source_video_path"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
].copy()
if len(missing_source_video) > 0:
    atomic_write_csv(
        missing_source_video,
        DIRS["metrics"] / "metadata_missing_source_video.csv",
    )
    raise RuntimeError(
        f"{len(missing_source_video)} eşleşmiş satırda orijinal video yolu boş. "
        "Ayrıntı metrics/metadata_missing_source_video.csv içinde."
    )

label_mismatch = valid[
    valid[CONFIG["label_column"]]
    != valid["selection_label"]
]
split_mismatch = valid[
    valid[CONFIG["split_column"]]
    != valid["selection_split"]
]

if len(label_mismatch) > 0:
    raise AssertionError(
        f"ROI ve seçim metadata arasında {len(label_mismatch)} label uyuşmazlığı var."
    )
if len(split_mismatch) > 0:
    raise AssertionError(
        f"ROI ve seçim metadata arasında {len(split_mismatch)} split uyuşmazlığı var."
    )

valid["target"] = (
    valid[CONFIG["label_column"]]
    .map({"real": 0, "fake": 1})
    .astype(int)
)

# Label prefix prevents a real video number and a fake sequence name
# from accidentally sharing the same group key.
valid["video_id"] = (
    valid[CONFIG["label_column"]]
    + "__"
    + valid["source_video_name"].astype(str)
)

if not valid[CONFIG["sample_id_column"]].is_unique:
    duplicate_count = int(
        valid[CONFIG["sample_id_column"]].duplicated().sum()
    )
    raise AssertionError(
        f"sample_id benzersiz değil. Tekrarlı satır: {duplicate_count}"
    )

# Video leakage checks.
video_split_counts = valid.groupby("video_id")[
    CONFIG["split_column"]
].nunique()
if int(video_split_counts.max()) > 1:
    leaking = video_split_counts[
        video_split_counts > 1
    ].index.tolist()[:20]
    raise AssertionError(
        f"Aynı video birden fazla split içinde: {leaking}"
    )

video_target_counts = valid.groupby("video_id")[
    "target"
].nunique()
if int(video_target_counts.max()) > 1:
    bad_videos = video_target_counts[
        video_target_counts > 1
    ].index.tolist()[:20]
    raise AssertionError(
        f"Aynı video_id altında birden fazla sınıf var: {bad_videos}"
    )

video_sets = {
    split: set(
        valid.loc[
            valid[CONFIG["split_column"]] == split,
            "video_id",
        ]
    )
    for split in ["train", "val", "test"]
}
assert video_sets["train"].isdisjoint(video_sets["val"])
assert video_sets["train"].isdisjoint(video_sets["test"])
assert video_sets["val"].isdisjoint(video_sets["test"])

valid = valid.sort_values(
    [CONFIG["split_column"], CONFIG["label_column"], "frame_name"]
).reset_index(drop=True)

atomic_write_csv(
    valid,
    DIRS["artifacts"] / "metadata_used.csv",
)

accounting = {
    "roi_metadata_total_rows": int(total_roi_rows),
    "successful_rows_before_file_check": int(len(successful_roi) + len(missing_success_files)),
    "missing_success_files": int(len(missing_success_files)),
    "final_usable_samples": int(len(valid)),
    "split_counts": {
        split: int(
            (valid[CONFIG["split_column"]] == split).sum()
        )
        for split in ["train", "val", "test"]
    },
    "class_counts": {
        label: int(
            (valid[CONFIG["label_column"]] == label).sum()
        )
        for label in ["real", "fake"]
    },
    "video_counts": {
        split: int(
            valid.loc[
                valid[CONFIG["split_column"]] == split,
                "video_id",
            ].nunique()
        )
        for split in ["train", "val", "test"]
    },
}
atomic_write_json(
    accounting,
    DIRS["metrics"] / "data_accounting.json",
)

initial_quality_gates = {
    "metadata_columns": "PASSED",
    "successful_file_existence": "PASSED",
    "selection_metadata_match": "PASSED",
    "label_consistency": "PASSED",
    "split_consistency": "PASSED",
    "sample_id_uniqueness": "PASSED",
    "video_split_leakage": "PASSED",
    "video_label_consistency": "PASSED",
}
atomic_write_json(
    initial_quality_gates,
    DIRS["metrics"] / "quality_gates_initial.json",
)

log(f"Toplam kaş metadata satırı: {total_roi_rows}")
log(f"Kullanılacak başarılı kaş frame sayısı: {len(valid)}")
log(f"Split dağılımı: {accounting['split_counts']}")
log(f"Sınıf dağılımı: {accounting['class_counts']}")
display(
    valid.groupby(
        [CONFIG["split_column"], CONFIG["label_column"]]
    ).size().rename("sample_count").to_frame()
)

2026-08-06 21:25:04,811 | INFO | Toplam kaş metadata satırı: 3000


INFO:eyebrow_experiment:Toplam kaş metadata satırı: 3000


2026-08-06 21:25:04,813 | INFO | Kullanılacak başarılı kaş frame sayısı: 1962


INFO:eyebrow_experiment:Kullanılacak başarılı kaş frame sayısı: 1962


2026-08-06 21:25:04,817 | INFO | Split dağılımı: {'train': 1562, 'val': 204, 'test': 196}


INFO:eyebrow_experiment:Split dağılımı: {'train': 1562, 'val': 204, 'test': 196}


2026-08-06 21:25:04,820 | INFO | Sınıf dağılımı: {'real': 993, 'fake': 969}


INFO:eyebrow_experiment:Sınıf dağılımı: {'real': 993, 'fake': 969}


sample_count
split label              
test  fake             95
      real            101
train fake            776
      real            786
val   fake             98
      real            106

In [7]:
# ============================================================
# 6. DATASET DISTRIBUTION FIGURE
# ============================================================
def save_figure(fig: plt.Figure, filename: str) -> Path:
    target = DIRS["figures"] / filename
    fig.savefig(
        target,
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)
    with Image.open(target) as image:
        if min(image.size) < 600:
            raise AssertionError(
                f"Figure resolution is below 600 px: {image.size}"
            )
    return target

split_class = (
    valid.groupby(
        [CONFIG["split_column"], CONFIG["label_column"]]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
split_class.plot(kind="bar", ax=ax)
ax.set_title("Eyebrow ROI Class Distribution by Dataset Split")
ax.set_xlabel("Dataset Split")
ax.set_ylabel("Number of Samples")
ax.legend(title="Class")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, "dataset_distribution.png")

print("Dataset distribution figure saved.")

Dataset distribution figure saved.


In [8]:
# ============================================================
# 7. HANDCRAFTED FEATURE EXTRACTION
# LBP + GLCM + Gabor + Wavelet
# ============================================================
def read_gray_image(path: str, size: int = 224) -> np.ndarray:
    image = Image.open(path).convert("L")
    image = ImageOps.pad(
        image,
        (size, size),
        method=Image.Resampling.BILINEAR,
        color=0,
        centering=(0.5, 0.5),
    )
    return np.asarray(image, dtype=np.uint8)

def safe_entropy(values: np.ndarray, bins: int = 64) -> float:
    flat = np.asarray(values, dtype=np.float64).ravel()
    if flat.size == 0:
        return 0.0
    hist, _ = np.histogram(flat, bins=bins, density=False)
    probs = hist.astype(np.float64)
    total = probs.sum()
    if total <= 0:
        return 0.0
    probs /= total
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())

def extract_lbp(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    features = []
    names = []
    for radius, points in zip(
        CONFIG["lbp_radii"],
        CONFIG["lbp_points"],
    ):
        lbp = local_binary_pattern(
            gray,
            P=points,
            R=radius,
            method="uniform",
        )
        bins = points + 2
        hist, _ = np.histogram(
            lbp.ravel(),
            bins=np.arange(0, bins + 1),
            range=(0, bins),
        )
        hist = hist.astype(np.float32)
        hist /= hist.sum() + 1e-8
        features.extend(hist.tolist())
        names.extend([
            f"lbp_r{radius}_p{points}_bin{index}"
            for index in range(bins)
        ])
    return np.asarray(features, dtype=np.float32), names

def extract_glcm(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    levels = int(CONFIG["glcm_levels"])
    quantized = np.floor(
        gray.astype(np.float32) / 256.0 * levels
    )
    quantized = np.clip(
        quantized,
        0,
        levels - 1,
    ).astype(np.uint8)

    angles = np.deg2rad(CONFIG["glcm_angles_deg"])
    matrix = graycomatrix(
        quantized,
        distances=CONFIG["glcm_distances"],
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True,
    )

    properties = [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM",
    ]
    features = []
    names = []
    for prop in properties:
        values = graycoprops(matrix, prop)
        for distance_index, distance in enumerate(
            CONFIG["glcm_distances"]
        ):
            for angle_index, angle in enumerate(
                CONFIG["glcm_angles_deg"]
            ):
                features.append(
                    float(values[distance_index, angle_index])
                )
                names.append(
                    f"glcm_{prop}_d{distance}_a{angle}"
                )
    return np.asarray(features, dtype=np.float32), names

def extract_gabor(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    image = gray.astype(np.float32) / 255.0
    features = []
    names = []

    for frequency in CONFIG["gabor_frequencies"]:
        for angle_deg in CONFIG["gabor_orientations_deg"]:
            theta = np.deg2rad(angle_deg)
            real, imaginary = gabor(
                image,
                frequency=frequency,
                theta=theta,
            )
            magnitude = np.sqrt(
                real ** 2 + imaginary ** 2
            )
            stats = {
                "mean": float(magnitude.mean()),
                "std": float(magnitude.std()),
                "energy": float(np.mean(magnitude ** 2)),
                "entropy": safe_entropy(magnitude),
            }
            for stat_name, value in stats.items():
                features.append(value)
                names.append(
                    f"gabor_f{frequency:.2f}_a{angle_deg}_{stat_name}"
                )

    return np.asarray(features, dtype=np.float32), names

def coefficient_statistics(
    coefficients: np.ndarray,
    prefix: str,
) -> Tuple[List[float], List[str]]:
    array = np.asarray(coefficients, dtype=np.float64)
    absolute = np.abs(array)
    stats = {
        "mean": float(array.mean()),
        "std": float(array.std()),
        "energy": float(np.mean(array ** 2)),
        "abs_mean": float(absolute.mean()),
        "abs_max": float(absolute.max()),
        "entropy": safe_entropy(array),
    }
    return (
        list(stats.values()),
        [f"{prefix}_{name}" for name in stats],
    )

def extract_wavelet(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    image = gray.astype(np.float32) / 255.0
    coefficients = pywt.wavedec2(
        image,
        wavelet=CONFIG["wavelet"],
        level=CONFIG["wavelet_level"],
        mode="symmetric",
    )

    features = []
    names = []

    approximation = coefficients[0]
    values, value_names = coefficient_statistics(
        approximation,
        f"wavelet_L{CONFIG['wavelet_level']}_LL",
    )
    features.extend(values)
    names.extend(value_names)

    current_level = CONFIG["wavelet_level"]
    for horizontal, vertical, diagonal in coefficients[1:]:
        for band_name, band in [
            ("LH", horizontal),
            ("HL", vertical),
            ("HH", diagonal),
        ]:
            values, value_names = coefficient_statistics(
                band,
                f"wavelet_L{current_level}_{band_name}",
            )
            features.extend(values)
            names.extend(value_names)
        current_level -= 1

    return np.asarray(features, dtype=np.float32), names

def extract_texture_features(
    path: str,
) -> Tuple[np.ndarray, Dict[str, slice], List[str]]:
    gray = read_gray_image(
        path,
        CONFIG["image_size"],
    )

    groups = {}
    all_features = []
    all_names = []
    start = 0

    for group_name, extractor in [
        ("lbp", extract_lbp),
        ("glcm", extract_glcm),
        ("gabor", extract_gabor),
        ("wavelet", extract_wavelet),
    ]:
        values, names = extractor(gray)
        end = start + len(values)
        groups[group_name] = slice(start, end)
        start = end
        all_features.append(values)
        all_names.extend(names)

    vector = np.concatenate(
        all_features
    ).astype(np.float32)

    if not np.isfinite(vector).all():
        raise FloatingPointError(
            f"NaN/Inf texture feature detected: {path}"
        )

    return vector, groups, all_names

example_vector, FEATURE_SLICES, FEATURE_NAMES = (
    extract_texture_features(
        valid.iloc[0]["resolved_image_path"]
    )
)

FEATURE_COUNTS = {
    name: int(group_slice.stop - group_slice.start)
    for name, group_slice in FEATURE_SLICES.items()
}

assert len(example_vector) == 240, (
    f"Beklenen texture boyutu 240, bulunan: {len(example_vector)}"
)

atomic_write_json(
    {
        "total_texture_dimension": int(len(example_vector)),
        "feature_counts": FEATURE_COUNTS,
        "feature_slices": {
            name: [group_slice.start, group_slice.stop]
            for name, group_slice in FEATURE_SLICES.items()
        },
    },
    DIRS["artifacts"] / "feature_dimensions.json",
)
atomic_write_text(
    "\n".join(FEATURE_NAMES),
    DIRS["artifacts"] / "feature_names.txt",
)

print("Feature counts:", FEATURE_COUNTS)
print("Total handcrafted texture dimension:", len(example_vector))

Feature counts: {'lbp': 54, 'glcm': 72, 'gabor': 72, 'wavelet': 42}
Total handcrafted texture dimension: 240


In [9]:
# ============================================================
# 8. BUILD OR RESUME TEXTURE FEATURE CACHE
# ============================================================
cache_key_payload = {
    "region": CONFIG["region"],
    "data_root": CONFIG["data_root"],
    "roi_metadata_path": CONFIG["roi_metadata_path"],
    "image_size": CONFIG["image_size"],
    "lbp_radii": CONFIG["lbp_radii"],
    "lbp_points": CONFIG["lbp_points"],
    "glcm_distances": CONFIG["glcm_distances"],
    "glcm_angles_deg": CONFIG["glcm_angles_deg"],
    "glcm_levels": CONFIG["glcm_levels"],
    "gabor_orientations_deg": CONFIG["gabor_orientations_deg"],
    "gabor_frequencies": CONFIG["gabor_frequencies"],
    "wavelet": CONFIG["wavelet"],
    "wavelet_level": CONFIG["wavelet_level"],
}
CACHE_VERSION = hashlib.sha256(
    json.dumps(
        cache_key_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:12]

CACHE_FILE = (
    DIRS["artifacts"]
    / f"texture_features_{CACHE_VERSION}.joblib"
)
CACHE_MANIFEST = (
    DIRS["artifacts"]
    / f"texture_manifest_{CACHE_VERSION}.csv"
)

if (
    CACHE_FILE.exists()
    and not CONFIG["overwrite_feature_cache"]
):
    texture_cache = joblib.load(CACHE_FILE)
else:
    texture_cache = {}

valid_sample_ids = set(
    valid[CONFIG["sample_id_column"]].astype(str)
)
# Remove any stale records not present in this run.
texture_cache = {
    key: value
    for key, value in texture_cache.items()
    if key in valid_sample_ids
}

pending_rows = valid[
    ~valid[CONFIG["sample_id_column"]]
    .astype(str)
    .isin(texture_cache.keys())
]

log(f"Cached texture samples: {len(texture_cache)}")
log(f"Pending texture samples: {len(pending_rows)}")

cache_errors = []
processed_since_flush = 0

for _, row in tqdm(
    pending_rows.iterrows(),
    total=len(pending_rows),
    desc="Extracting eyebrow texture features",
):
    sample_id = str(
        row[CONFIG["sample_id_column"]]
    )
    image_path = row["resolved_image_path"]

    try:
        vector, _, _ = extract_texture_features(
            image_path
        )
        texture_cache[sample_id] = vector
    except Exception as exc:
        cache_errors.append({
            "sample_id": sample_id,
            "image_path": image_path,
            "error_type": type(exc).__name__,
            "error": str(exc),
        })

    processed_since_flush += 1
    if (
        processed_since_flush
        >= CONFIG["feature_cache_flush_every"]
    ):
        atomic_joblib_dump(
            texture_cache,
            CACHE_FILE,
        )
        processed_since_flush = 0

atomic_joblib_dump(texture_cache, CACHE_FILE)

if cache_errors:
    atomic_write_csv(
        pd.DataFrame(cache_errors),
        DIRS["metrics"] / "texture_feature_errors.csv",
    )

valid["texture_cached"] = (
    valid[CONFIG["sample_id_column"]]
    .astype(str)
    .isin(texture_cache.keys())
)
feature_manifest = valid[
    valid["texture_cached"]
].copy()
atomic_write_csv(
    feature_manifest,
    CACHE_MANIFEST,
)

valid = feature_manifest.reset_index(drop=True)
if valid.empty:
    raise RuntimeError(
        "Texture feature extraction produced no valid sample."
    )

log(f"Final cached eyebrow samples used: {len(valid)}")

2026-08-06 21:25:26,031 | INFO | Cached texture samples: 0


INFO:eyebrow_experiment:Cached texture samples: 0


2026-08-06 21:25:26,033 | INFO | Pending texture samples: 1962


INFO:eyebrow_experiment:Pending texture samples: 1962


Extracting eyebrow texture features:   0%|          | 0/1962 [00:00<?, ?it/s]

2026-08-06 22:25:38,432 | INFO | Final cached eyebrow samples used: 1962


INFO:eyebrow_experiment:Final cached eyebrow samples used: 1962


In [10]:
# ============================================================
# 9. TRAIN-ONLY TEXTURE SCALER + FEATURE ARRAY ARTIFACTS
# ============================================================
train_ids = valid.loc[
    valid[CONFIG["split_column"]] == "train",
    CONFIG["sample_id_column"],
].astype(str)

train_texture = np.stack([
    texture_cache[sample_id]
    for sample_id in train_ids
])

texture_scaler = StandardScaler()
texture_scaler.fit(train_texture)

SCALER_FILE = (
    DIRS["artifacts"] / "texture_scaler.joblib"
)
atomic_joblib_dump(
    texture_scaler,
    SCALER_FILE,
)

atomic_write_json(
    {
        "fit_split": "train",
        "feature_dimension": int(
            train_texture.shape[1]
        ),
        "cache_version": CACHE_VERSION,
        "feature_groups": {
            name: [
                group_slice.start,
                group_slice.stop,
            ]
            for name, group_slice in FEATURE_SLICES.items()
        },
    },
    DIRS["artifacts"] / "texture_feature_schema.json",
)

if CONFIG["save_feature_arrays"]:
    for split in ["train", "val", "test"]:
        split_rows = valid[
            valid[CONFIG["split_column"]] == split
        ].copy()
        sample_ids = (
            split_rows[CONFIG["sample_id_column"]]
            .astype(str)
            .to_numpy()
        )
        raw_features = np.stack([
            texture_cache[sample_id]
            for sample_id in sample_ids
        ]).astype(np.float32)
        scaled_features = texture_scaler.transform(
            raw_features
        ).astype(np.float32)

        np.savez_compressed(
            DIRS["artifacts"]
            / f"texture_features_{split}.npz",
            sample_ids=sample_ids,
            video_ids=split_rows["video_id"].astype(str).to_numpy(),
            labels=split_rows["target"].to_numpy(dtype=np.int64),
            raw_features=raw_features,
            scaled_features=scaled_features,
        )

print("Texture scaler fitted only on the training split.")

Texture scaler fitted only on the training split.


In [11]:
# ============================================================
# 10. DATASET AND TRANSFORMS
# ============================================================
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Image and precomputed texture branches must describe the same
# unaugmented ROI. Therefore no geometric augmentation is used.
image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])

MODE_GROUPS = {
    "swin_only": [],
    "texture_only": [
        "lbp",
        "glcm",
        "gabor",
        "wavelet",
    ],
    "swin_lbp": ["lbp"],
    "swin_lbp_glcm": [
        "lbp",
        "glcm",
    ],
    "swin_lbp_glcm_gabor": [
        "lbp",
        "glcm",
        "gabor",
    ],
    "full": [
        "lbp",
        "glcm",
        "gabor",
        "wavelet",
    ],
}

def select_texture_groups(
    vector: np.ndarray,
    mode: str,
) -> np.ndarray:
    groups = MODE_GROUPS[mode]
    if not groups:
        return np.zeros((0,), dtype=np.float32)
    selected = [
        vector[FEATURE_SLICES[group]]
        for group in groups
    ]
    return np.concatenate(
        selected
    ).astype(np.float32)

def prepare_rgb_image(path: str) -> Image.Image:
    image = Image.open(path).convert("RGB")
    return ImageOps.pad(
        image,
        (
            CONFIG["image_size"],
            CONFIG["image_size"],
        ),
        method=Image.Resampling.BILINEAR,
        color=(0, 0, 0),
        centering=(0.5, 0.5),
    )

class EyebrowFusionDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        mode: str,
        scaler: StandardScaler,
    ):
        self.df = dataframe.reset_index(
            drop=True
        ).copy()
        self.mode = mode
        self.scaler = scaler

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(
        self,
        index: int,
    ) -> Dict[str, Any]:
        row = self.df.iloc[index]
        image_path = row["resolved_image_path"]
        sample_id = str(
            row[CONFIG["sample_id_column"]]
        )

        image = prepare_rgb_image(image_path)
        image_tensor = image_transform(image)

        raw_texture = texture_cache[
            sample_id
        ].reshape(1, -1)
        scaled_texture = self.scaler.transform(
            raw_texture
        )[0].astype(np.float32)
        selected_texture = select_texture_groups(
            scaled_texture,
            self.mode,
        )

        return {
            "image": image_tensor,
            "texture": torch.from_numpy(
                selected_texture
            ),
            "target": torch.tensor(
                int(row["target"]),
                dtype=torch.float32,
            ),
            "sample_id": sample_id,
            "video_id": str(row["video_id"]),
            "image_path": image_path,
            "source_video_path": str(
                row["source_video_path"]
            ),
        }

def build_loaders(mode: str):
    split_frames = {
        split: valid[
            valid[CONFIG["split_column"]] == split
        ].copy()
        for split in ["train", "val", "test"]
    }

    datasets = {
        split: EyebrowFusionDataset(
            split_frames[split],
            mode,
            texture_scaler,
        )
        for split in ["train", "val", "test"]
    }

    generator = torch.Generator()
    generator.manual_seed(CONFIG["seed"])

    train_size = len(datasets["train"])
    if train_size < 2:
        raise RuntimeError(
            "Train split en az 2 örnek içermeli; sınıflandırıcıdaki "
            "BatchNorm1d tek örnekli batch ile eğitilemez."
        )

    train_batch_size = min(
        int(CONFIG["batch_size"]),
        train_size,
    )
    # Yalnızca son batch tek örnek kalacaksa düşür. Böylece küçük veri
    # setlerinde loader tamamen boş kalmaz, BatchNorm hatası da oluşmaz.
    train_drop_last = (
        train_size % train_batch_size == 1
    )

    loaders = {
        "train": DataLoader(
            datasets["train"],
            batch_size=train_batch_size,
            shuffle=True,
            num_workers=CONFIG["num_workers"],
            pin_memory=(DEVICE.type == "cuda"),
            generator=generator,
            drop_last=train_drop_last,
        ),
        "val": DataLoader(
            datasets["val"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=(DEVICE.type == "cuda"),
        ),
        "test": DataLoader(
            datasets["test"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=(DEVICE.type == "cuda"),
        ),
    }

    return datasets, loaders

print("Eyebrow dataset and transform pipeline ready.")

Eyebrow dataset and transform pipeline ready.


In [12]:
# ============================================================
# 11. SWIN V2-TINY + TEXTURE FUSION MODEL
# ============================================================
class SwinTextureFusion(nn.Module):
    def __init__(
        self,
        texture_dim: int,
        mode: str,
        hidden_dim: int,
        dropout: float,
        pretrained: bool = True,
    ):
        super().__init__()
        self.mode = mode
        self.use_swin = mode != "texture_only"
        self.use_texture = mode != "swin_only"

        self.swin_dim = 0
        if self.use_swin:
            selected_weights = (
                Swin_V2_T_Weights.DEFAULT
                if pretrained
                else None
            )
            self.backbone = swin_v2_t(
                weights=selected_weights
            )
            self.swin_dim = (
                self.backbone.head.in_features
            )
            self.backbone.head = nn.Identity()
            self.swin_projection = nn.Sequential(
                nn.LayerNorm(self.swin_dim),
                nn.Linear(
                    self.swin_dim,
                    hidden_dim,
                ),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        else:
            self.backbone = None
            self.swin_projection = None

        self.texture_dim = texture_dim
        if self.use_texture:
            self.texture_projection = nn.Sequential(
                nn.LayerNorm(texture_dim),
                nn.Linear(
                    texture_dim,
                    hidden_dim,
                ),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        else:
            self.texture_projection = None

        fusion_dim = (
            hidden_dim * int(self.use_swin)
            + hidden_dim * int(self.use_texture)
        )

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dim,
                hidden_dim // 2,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dim // 2,
                1,
            ),
        )

    def forward(
        self,
        image: torch.Tensor,
        texture: torch.Tensor,
    ) -> torch.Tensor:
        branches = []

        if self.use_swin:
            deep_features = self.backbone(image)
            branches.append(
                self.swin_projection(deep_features)
            )

        if self.use_texture:
            branches.append(
                self.texture_projection(texture)
            )

        fused = torch.cat(branches, dim=1)
        return self.classifier(
            fused
        ).squeeze(1)

    def freeze_backbone(self) -> None:
        if self.backbone is not None:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

    def unfreeze_backbone_tail(self) -> None:
        if self.backbone is None:
            return

        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        for parameter in self.backbone.features[-1].parameters():
            parameter.requires_grad = True
        for parameter in self.backbone.norm.parameters():
            parameter.requires_grad = True
        for parameter in self.swin_projection.parameters():
            parameter.requires_grad = True

def texture_dimension_for_mode(mode: str) -> int:
    dummy = np.zeros(
        len(FEATURE_NAMES),
        dtype=np.float32,
    )
    return len(
        select_texture_groups(dummy, mode)
    )

print("Model definition ready.")

Model definition ready.


In [13]:
# ============================================================
# 12. METRICS, THRESHOLD, OPTIMIZER AND CHECKPOINT HELPERS
# ============================================================
def binary_metrics(
    targets: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    predictions = (
        probabilities >= threshold
    ).astype(int)

    result = {
        "accuracy": accuracy_score(
            targets,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            targets,
            predictions,
        ),
        "precision": precision_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            targets,
            predictions,
            zero_division=0,
        ),
    }

    if len(np.unique(targets)) == 2:
        result["roc_auc"] = roc_auc_score(
            targets,
            probabilities,
        )
        result["average_precision"] = (
            average_precision_score(
                targets,
                probabilities,
            )
        )
    else:
        result["roc_auc"] = float("nan")
        result["average_precision"] = float("nan")

    return {
        key: float(value)
        for key, value in result.items()
    }

def find_best_threshold(
    targets: np.ndarray,
    probabilities: np.ndarray,
) -> Tuple[float, float, pd.DataFrame]:
    thresholds = np.linspace(
        CONFIG["threshold_search_min"],
        CONFIG["threshold_search_max"],
        CONFIG["threshold_search_steps"],
    )

    rows = []
    best_threshold = CONFIG["threshold"]
    best_f1 = -np.inf

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)
        score = f1_score(
            targets,
            predictions,
            zero_division=0,
        )
        rows.append({
            "threshold": float(threshold),
            "f1": float(score),
        })
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)

    return (
        best_threshold,
        best_f1,
        pd.DataFrame(rows),
    )

def build_optimizer(model: SwinTextureFusion):
    backbone_parameters = []
    head_parameters = []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("backbone."):
            backbone_parameters.append(parameter)
        else:
            head_parameters.append(parameter)

    parameter_groups = []
    if backbone_parameters:
        parameter_groups.append({
            "params": backbone_parameters,
            "lr": CONFIG["backbone_learning_rate"],
        })
    if head_parameters:
        parameter_groups.append({
            "params": head_parameters,
            "lr": CONFIG["learning_rate"],
        })

    if not parameter_groups:
        raise RuntimeError(
            "Optimizer için eğitilebilir parametre bulunamadı."
        )

    return torch.optim.AdamW(
        parameter_groups,
        weight_decay=CONFIG["weight_decay"],
    )

def capture_rng_state() -> Dict[str, Any]:
    state = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda_rng_state"] = (
            torch.cuda.get_rng_state_all()
        )
    return state

def checkpoint_state(
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    best_metric: float,
    history: List[Dict[str, float]],
    mode: str,
    best_threshold: float,
) -> Dict[str, Any]:
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_metric_score": float(best_metric),
        "best_threshold": float(best_threshold),
        "selection_metric": "val_roc_auc",
        "history": history,
        "config": CONFIG_RESOLVED,
        "mode": mode,
        "feature_slices": {
            name: [
                group_slice.start,
                group_slice.stop,
            ]
            for name, group_slice in FEATURE_SLICES.items()
        },
    }
    state.update(capture_rng_state())
    return state

print("Metric and checkpoint helpers ready.")

Metric and checkpoint helpers ready.


In [14]:
# ============================================================
# 13. TRAIN AND PREDICTION LOOPS
# ============================================================
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer=None,
    scaler=None,
    training: bool = False,
    max_batches: Optional[int] = None,
) -> Tuple[float, np.ndarray, np.ndarray]:
    model.train(training)
    losses = []
    targets_all = []
    probabilities_all = []

    for batch_index, batch in enumerate(loader):
        if (
            max_batches is not None
            and batch_index >= max_batches
        ):
            break

        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )
        textures = batch["texture"].to(
            DEVICE,
            non_blocking=True,
        )
        targets = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            enabled=(
                CONFIG["amp"]
                and DEVICE.type == "cuda"
            ),
        ):
            logits = model(images, textures)
            loss = criterion(logits, targets)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss at batch {batch_index}"
            )

        if training:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CONFIG["gradient_clip_norm"],
            )
            scaler.step(optimizer)
            scaler.update()

        losses.append(float(loss.detach().cpu()))
        probabilities = (
            torch.sigmoid(logits)
            .detach()
            .cpu()
            .numpy()
        )
        probabilities_all.extend(
            probabilities.tolist()
        )
        targets_all.extend(
            targets.detach().cpu().numpy().tolist()
        )

    if not losses:
        raise RuntimeError(
            "DataLoader hiçbir batch üretmedi."
        )

    return (
        float(np.mean(losses)),
        np.asarray(
            targets_all,
            dtype=np.int64,
        ),
        np.asarray(
            probabilities_all,
            dtype=np.float64,
        ),
    )

def predict_loader(
    model: nn.Module,
    loader: DataLoader,
) -> pd.DataFrame:
    model.eval()
    records = []

    with torch.no_grad():
        for batch in tqdm(
            loader,
            desc="Predicting test eyebrow frames",
        ):
            images = batch["image"].to(
                DEVICE,
                non_blocking=True,
            )
            textures = batch["texture"].to(
                DEVICE,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=(
                    CONFIG["amp"]
                    and DEVICE.type == "cuda"
                ),
            ):
                logits = model(images, textures)

            probabilities = (
                torch.sigmoid(logits)
                .cpu()
                .numpy()
            )
            targets = batch["target"].numpy()

            for index in range(
                len(probabilities)
            ):
                records.append({
                    "sample_id": batch["sample_id"][index],
                    "video_id": batch["video_id"][index],
                    "image_path": batch["image_path"][index],
                    "source_video_path": batch[
                        "source_video_path"
                    ][index],
                    "target": int(targets[index]),
                    "probability_fake": float(
                        probabilities[index]
                    ),
                })

    return pd.DataFrame(records)

In [15]:
# ============================================================
# 14. SMOKE TEST
# ============================================================
def smoke_test(mode: str) -> None:
    datasets, loaders = build_loaders(mode)
    texture_dim = texture_dimension_for_mode(mode)

    model = SwinTextureFusion(
        texture_dim=texture_dim,
        mode=mode,
        hidden_dim=CONFIG["hidden_dim"],
        dropout=CONFIG["dropout"],
        # Smoke test yalnızca veri akışı ve tensor boyutlarını doğrular.
        # Asıl eğitim pretrained ağırlıkları bir kez yükler.
        pretrained=False,
    ).to(DEVICE)

    if (
        model.use_swin
        and CONFIG["freeze_backbone_epochs"] > 0
    ):
        model.freeze_backbone()

    optimizer = build_optimizer(model)
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, CONFIG["epochs"]),
        )
    )

    train_targets = (
        datasets["train"].df["target"].to_numpy()
    )
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=train_targets,
    )
    positive_weight = torch.tensor(
        [class_weights[1] / class_weights[0]],
        device=DEVICE,
        dtype=torch.float32,
    )
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=positive_weight
    )
    grad_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            CONFIG["amp"]
            and DEVICE.type == "cuda"
        ),
    )

    train_loss, _, _ = run_epoch(
        model,
        loaders["train"],
        criterion,
        optimizer=optimizer,
        scaler=grad_scaler,
        training=True,
        max_batches=CONFIG["smoke_train_batches"],
    )
    val_loss, _, _ = run_epoch(
        model,
        loaders["val"],
        criterion,
        training=False,
        max_batches=CONFIG["smoke_val_batches"],
    )

    smoke_checkpoint = (
        DIRS["checkpoints"] / "smoke_test.ckpt"
    )
    state = checkpoint_state(
        epoch=0,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=grad_scaler,
        best_metric=0.0,
        history=[],
        mode=mode,
        best_threshold=CONFIG["threshold"],
    )
    atomic_torch_save(
        state,
        smoke_checkpoint,
    )
    _ = torch.load(
        smoke_checkpoint,
        map_location="cpu",
        weights_only=False,
    )
    smoke_checkpoint.unlink(missing_ok=True)

    del model, optimizer, loaders, datasets
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    log(
        f"Smoke test passed | mode={mode} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_loss:.5f}"
    )

if CONFIG["smoke_test"]:
    smoke_test(CONFIG["active_mode"])
else:
    log("Smoke test disabled.")

2026-08-06 22:26:14,387 | INFO | Smoke test passed | mode=full | train_loss=0.69888 | val_loss=0.73660


INFO:eyebrow_experiment:Smoke test passed | mode=full | train_loss=0.69888 | val_loss=0.73660


In [16]:
# ============================================================
# 15. TRAIN EXPERIMENT
# Main run outputs use the exact root folder format requested.
# ============================================================
def directories_for_mode(
    mode: str,
) -> Dict[str, Path]:
    # The active/default model writes directly into the requested
    # checkpoints/logs/metrics/predictions/figures/artifacts folders.
    if (
        not CONFIG["run_all_ablations"]
        and mode == CONFIG["active_mode"]
    ):
        return DIRS

    # Optional ablations are isolated under artifacts/ablations.
    root = (
        DIRS["artifacts"]
        / "ablations"
        / mode
    )
    mode_dirs = {
        "run": root,
        "checkpoints": root / "checkpoints",
        "logs": root / "logs",
        "metrics": root / "metrics",
        "predictions": root / "predictions",
        "figures": root / "figures",
        "artifacts": root / "artifacts",
    }
    for path in mode_dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return mode_dirs

def train_experiment(
    mode: str,
) -> Dict[str, Any]:
    log("=" * 70)
    log(f"Training mode: {mode}")
    log("=" * 70)

    mode_dirs = directories_for_mode(mode)
    datasets, loaders = build_loaders(mode)
    texture_dim = texture_dimension_for_mode(mode)

    model = SwinTextureFusion(
        texture_dim=texture_dim,
        mode=mode,
        hidden_dim=CONFIG["hidden_dim"],
        dropout=CONFIG["dropout"],
        pretrained=CONFIG["pretrained"],
    ).to(DEVICE)

    architecture_payload = {
        "region": CONFIG["region"],
        "mode": mode,
        "backbone": (
            "swin_v2_t"
            if model.use_swin
            else None
        ),
        "pretrained": bool(
            CONFIG["pretrained"]
        ),
        "swin_output_dimension": int(
            model.swin_dim
        ),
        "texture_dimension": int(
            texture_dim
        ),
        "hidden_dimension": int(
            CONFIG["hidden_dim"]
        ),
        "dropout": float(
            CONFIG["dropout"]
        ),
        "feature_groups": MODE_GROUPS[mode],
        "model_string": str(model),
    }
    atomic_write_json(
        architecture_payload,
        mode_dirs["artifacts"]
        / "model_architecture.json",
    )

    if (
        model.use_swin
        and CONFIG["freeze_backbone_epochs"] > 0
    ):
        model.freeze_backbone()

    train_targets = (
        datasets["train"].df["target"].to_numpy()
    )
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=train_targets,
    )
    positive_weight = torch.tensor(
        [class_weights[1] / class_weights[0]],
        device=DEVICE,
        dtype=torch.float32,
    )
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=positive_weight
    )

    optimizer = build_optimizer(model)
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, CONFIG["epochs"]),
        )
    )
    grad_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            CONFIG["amp"]
            and DEVICE.type == "cuda"
        ),
    )

    history = []
    best_auc = -np.inf
    best_threshold = CONFIG["threshold"]
    patience_counter = 0
    best_threshold_table = None

    for epoch in range(CONFIG["epochs"]):
        if (
            model.use_swin
            and CONFIG["unfreeze_last_stages"]
            and epoch == CONFIG["freeze_backbone_epochs"]
        ):
            model.unfreeze_backbone_tail()
            optimizer = build_optimizer(model)
            scheduler = (
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer,
                    T_max=max(
                        1,
                        CONFIG["epochs"] - epoch,
                    ),
                )
            )
            log("Final Swin stage unfrozen.")

        train_loss, train_y, train_p = run_epoch(
            model,
            loaders["train"],
            criterion,
            optimizer=optimizer,
            scaler=grad_scaler,
            training=True,
        )
        val_loss, val_y, val_p = run_epoch(
            model,
            loaders["val"],
            criterion,
            training=False,
        )

        train_metrics = binary_metrics(
            train_y,
            train_p,
            CONFIG["threshold"],
        )
        (
            epoch_threshold,
            epoch_threshold_f1,
            threshold_table,
        ) = find_best_threshold(
            val_y,
            val_p,
        )
        val_metrics = binary_metrics(
            val_y,
            val_p,
            epoch_threshold,
        )

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "validation_threshold": epoch_threshold,
            "validation_threshold_f1": epoch_threshold_f1,
            **{
                f"train_{key}": value
                for key, value in train_metrics.items()
            },
            **{
                f"val_{key}": value
                for key, value in val_metrics.items()
            },
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        history_frame = pd.DataFrame(history)
        atomic_write_csv(
            history_frame,
            mode_dirs["metrics"]
            / "epoch_metrics.csv",
        )
        atomic_write_csv(
            history_frame,
            mode_dirs["logs"]
            / "training_history.csv",
        )

        current_auc = val_metrics["roc_auc"]

        last_state = checkpoint_state(
            epoch=epoch + 1,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=grad_scaler,
            best_metric=best_auc,
            history=history,
            mode=mode,
            best_threshold=best_threshold,
        )
        atomic_torch_save(
            last_state,
            mode_dirs["checkpoints"]
            / "last.ckpt",
        )

        if (
            np.isfinite(current_auc)
            and current_auc > best_auc
        ):
            best_auc = current_auc
            best_threshold = epoch_threshold
            best_threshold_table = threshold_table.copy()
            patience_counter = 0

            best_state = checkpoint_state(
                epoch=epoch + 1,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=grad_scaler,
                best_metric=best_auc,
                history=history,
                mode=mode,
                best_threshold=best_threshold,
            )
            atomic_torch_save(
                best_state,
                mode_dirs["checkpoints"]
                / "best.ckpt",
            )
        else:
            patience_counter += 1

        scheduler.step()

        log(
            f"Epoch {epoch + 1:02d}/{CONFIG['epochs']} | "
            f"Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val F1 {val_metrics['f1']:.4f} | "
            f"Val AUC {val_metrics['roc_auc']:.4f} | "
            f"Threshold {epoch_threshold:.3f}"
        )

        if patience_counter >= CONFIG["patience"]:
            log("Early stopping activated.")
            break

    best_path = (
        mode_dirs["checkpoints"] / "best.ckpt"
    )
    if not best_path.exists():
        # Defensive fallback if validation AUC could not be computed.
        last_checkpoint = torch.load(
            mode_dirs["checkpoints"] / "last.ckpt",
            map_location="cpu",
            weights_only=False,
        )
        atomic_torch_save(
            last_checkpoint,
            best_path,
        )

    if best_threshold_table is not None:
        atomic_write_csv(
            best_threshold_table,
            mode_dirs["metrics"]
            / "validation_threshold_search.csv",
        )

    best_checkpoint = torch.load(
        best_path,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )
    selected_threshold = float(
        best_checkpoint.get(
            "best_threshold",
            CONFIG["threshold"],
        )
    )

    test_predictions = predict_loader(
        model,
        loaders["test"],
    )
    test_predictions["prediction"] = (
        test_predictions["probability_fake"]
        >= selected_threshold
    ).astype(int)
    test_predictions["correct"] = (
        test_predictions["prediction"]
        == test_predictions["target"]
    )

    atomic_write_csv(
        test_predictions,
        mode_dirs["predictions"]
        / "test_predictions_frame_level.csv",
    )

    test_metrics = binary_metrics(
        test_predictions["target"].to_numpy(),
        test_predictions[
            "probability_fake"
        ].to_numpy(),
        selected_threshold,
    )
    test_metrics.update({
        "evaluation_level": "frame",
        "mode": mode,
        "best_epoch": int(
            best_checkpoint["epoch"]
        ),
        "selection_metric": "val_roc_auc",
        "best_validation_auc": float(
            best_checkpoint[
                "best_metric_score"
            ]
        ),
        "selected_threshold": selected_threshold,
        "texture_dimension": int(texture_dim),
        "feature_groups": MODE_GROUPS[mode],
        "sample_count": int(
            len(test_predictions)
        ),
    })
    atomic_write_json(
        test_metrics,
        mode_dirs["metrics"]
        / "test_metrics_frame_level.json",
    )

    result = {
        "mode": mode,
        **test_metrics,
        "output_dir": str(
            mode_dirs["run"]
        ),
    }

    del model, optimizer, loaders, datasets
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [18]:
# ============================================================
# 16. RUN MAIN EXPERIMENT OR OPTIONAL ABLATIONS
# ============================================================
if CONFIG["run_all_ablations"]:
    modes_to_run = CONFIG["ablation_modes"]
else:
    modes_to_run = [CONFIG["active_mode"]]

experiment_results = []
for mode in modes_to_run:
    experiment_results.append(
        train_experiment(mode)
    )

results_frame = pd.DataFrame(
    experiment_results
)
atomic_write_csv(
    results_frame,
    DIRS["metrics"] / "ablation_results.csv",
)
display(results_frame)

2026-08-06 22:33:24,907 | INFO | ======================================================================


INFO:eyebrow_experiment:======================================================================


2026-08-06 22:33:24,910 | INFO | Training mode: full


INFO:eyebrow_experiment:Training mode: full


2026-08-06 22:33:24,913 | INFO | ======================================================================


...

INFO:eyebrow_experiment:Epoch 13/30 | Train Loss 0.6030 | Val Loss 0.6310 | Val F1 0.7037 | Val AUC 0.6943 | Threshold 0.300


2026-08-06 22:37:13,994 | INFO | Early stopping activated.


INFO:eyebrow_experiment:Early stopping activated.


Predicting test eyebrow frames:   0%|          | 0/13 [00:00<?, ?it/s]

,mode,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,evaluation_level,best_epoch,selection_metric,best_validation_auc,selected_threshold,texture_dimension,feature_groups,sample_count,output_dir
0,full,0.602041,0.609171,0.559441,0.842105,0.672269,0.668786,0.670791,frame,5,val_roc_auc,0.727955,0.33,240,"[lbp, glcm, gabor, wavelet]",196,/content/drive/MyDrive/AISC DeepFake Çalışmala...


In [19]:
# ============================================================
# 17. FRAME-LEVEL FIGURES
# ============================================================
history = pd.read_csv(
    DIRS["metrics"] / "epoch_metrics.csv"
)
frame_predictions = pd.read_csv(
    DIRS["predictions"]
    / "test_predictions_frame_level.csv"
)

# Loss curve
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    history["epoch"],
    history["train_loss"],
    label="Training Loss",
)
ax.plot(
    history["epoch"],
    history["val_loss"],
    label="Validation Loss",
)
ax.set_title("Training and Validation Loss Curve")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_figure(fig, "training_loss_curve.png")

# F1 curve
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    history["epoch"],
    history["train_f1"],
    label="Training F1",
)
ax.plot(
    history["epoch"],
    history["val_f1"],
    label="Validation F1",
)
ax.set_title("Training and Validation F1 Curve")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1 Score")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_figure(fig, "training_f1_curve.png")

# Frame-level confusion matrix
frame_cm = confusion_matrix(
    frame_predictions["target"],
    frame_predictions["prediction"],
    labels=[0, 1],
)
fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
image = ax.imshow(frame_cm)
for row_index in range(2):
    for column_index in range(2):
        ax.text(
            column_index,
            row_index,
            str(frame_cm[row_index, column_index]),
            ha="center",
            va="center",
        )
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])
ax.set_title("Test Confusion Matrix — Frame Level")
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
fig.colorbar(image, ax=ax)
fig.tight_layout()
save_figure(
    fig,
    "test_confusion_matrix_frame_level.png",
)

# Frame ROC
frame_fpr, frame_tpr, _ = roc_curve(
    frame_predictions["target"],
    frame_predictions["probability_fake"],
)
frame_auc = roc_auc_score(
    frame_predictions["target"],
    frame_predictions["probability_fake"],
)
fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
ax.plot(
    frame_fpr,
    frame_tpr,
    label=f"ROC AUC = {frame_auc:.4f}",
)
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier",
)
ax.set_title("Test ROC Curve — Frame Level")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_figure(
    fig,
    "test_roc_curve_frame_level.png",
)

# Frame precision-recall
frame_precision, frame_recall, _ = (
    precision_recall_curve(
        frame_predictions["target"],
        frame_predictions["probability_fake"],
    )
)
frame_ap = average_precision_score(
    frame_predictions["target"],
    frame_predictions["probability_fake"],
)
fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
ax.plot(
    frame_recall,
    frame_precision,
    label=f"Average Precision = {frame_ap:.4f}",
)
ax.set_title("Test Precision-Recall Curve — Frame Level")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_figure(
    fig,
    "test_precision_recall_curve_frame_level.png",
)

# Probability distribution
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
for target_value, class_name in [
    (0, "Real"),
    (1, "Fake"),
]:
    values = frame_predictions.loc[
        frame_predictions["target"] == target_value,
        "probability_fake",
    ]
    ax.hist(
        values,
        bins=30,
        alpha=0.55,
        label=class_name,
    )
ax.set_title("Test Fake-Probability Distribution — Frame Level")
ax.set_xlabel("Predicted Fake Probability")
ax.set_ylabel("Sample Count")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_figure(
    fig,
    "test_probability_distribution_frame_level.png",
)

print("Frame-level figures saved.")

Frame-level figures saved.


In [20]:
# ============================================================
# 18. VIDEO-LEVEL EVALUATION AND FIGURES
# ============================================================
frame_metrics = json.loads(
    (
        DIRS["metrics"]
        / "test_metrics_frame_level.json"
    ).read_text(encoding="utf-8")
)
selected_threshold = float(
    frame_metrics["selected_threshold"]
)

target_counts_per_video = (
    frame_predictions.groupby("video_id")[
        "target"
    ].nunique()
)
if int(target_counts_per_video.max()) > 1:
    raise AssertionError(
        "Aynı video_id altında birden fazla target bulundu."
    )

video_predictions = (
    frame_predictions.groupby(
        "video_id",
        as_index=False,
    )
    .agg(
        target=("target", "first"),
        probability_fake=(
            "probability_fake",
            "mean",
        ),
        sample_count=("sample_id", "count"),
        source_video_path=(
            "source_video_path",
            "first",
        ),
    )
)
video_predictions["prediction"] = (
    video_predictions["probability_fake"]
    >= selected_threshold
).astype(int)
video_predictions["correct"] = (
    video_predictions["prediction"]
    == video_predictions["target"]
)

atomic_write_csv(
    video_predictions,
    DIRS["predictions"]
    / "test_predictions_video_level.csv",
)

video_metrics = binary_metrics(
    video_predictions["target"].to_numpy(),
    video_predictions[
        "probability_fake"
    ].to_numpy(),
    selected_threshold,
)
video_metrics.update({
    "evaluation_level": "video",
    "mode": CONFIG["active_mode"],
    "selected_threshold": selected_threshold,
    "video_count": int(
        len(video_predictions)
    ),
    "frame_count": int(
        video_predictions["sample_count"].sum()
    ),
    "aggregation": "mean_probability",
})
atomic_write_json(
    video_metrics,
    DIRS["metrics"]
    / "test_metrics_video_level.json",
)

summary_rows = []
for level_name, metrics_payload in [
    ("frame", frame_metrics),
    ("video", video_metrics),
]:
    summary_rows.append({
        "evaluation_level": level_name,
        "accuracy": metrics_payload.get("accuracy"),
        "balanced_accuracy": metrics_payload.get(
            "balanced_accuracy"
        ),
        "precision": metrics_payload.get("precision"),
        "recall": metrics_payload.get("recall"),
        "f1": metrics_payload.get("f1"),
        "roc_auc": metrics_payload.get("roc_auc"),
        "average_precision": metrics_payload.get(
            "average_precision"
        ),
        "selected_threshold": selected_threshold,
        "sample_count": (
            frame_metrics.get("sample_count")
            if level_name == "frame"
            else video_metrics.get("video_count")
        ),
    })
atomic_write_csv(
    pd.DataFrame(summary_rows),
    DIRS["metrics"]
    / "test_metrics_summary.csv",
)

# Video confusion matrix
video_cm = confusion_matrix(
    video_predictions["target"],
    video_predictions["prediction"],
    labels=[0, 1],
)
fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
image = ax.imshow(video_cm)
for row_index in range(2):
    for column_index in range(2):
        ax.text(
            column_index,
            row_index,
            str(video_cm[row_index, column_index]),
            ha="center",
            va="center",
        )
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])
ax.set_title("Test Confusion Matrix — Video Level")
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
fig.colorbar(image, ax=ax)
fig.tight_layout()
save_figure(
    fig,
    "test_confusion_matrix_video_level.png",
)

# Video ROC and PR require two classes.
if video_predictions["target"].nunique() == 2:
    video_fpr, video_tpr, _ = roc_curve(
        video_predictions["target"],
        video_predictions["probability_fake"],
    )
    video_auc = roc_auc_score(
        video_predictions["target"],
        video_predictions["probability_fake"],
    )
    fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
    ax.plot(
        video_fpr,
        video_tpr,
        label=f"ROC AUC = {video_auc:.4f}",
    )
    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Random Classifier",
    )
    ax.set_title("Test ROC Curve — Video Level")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_figure(
        fig,
        "test_roc_curve_video_level.png",
    )

    video_precision, video_recall, _ = (
        precision_recall_curve(
            video_predictions["target"],
            video_predictions["probability_fake"],
        )
    )
    video_ap = average_precision_score(
        video_predictions["target"],
        video_predictions["probability_fake"],
    )
    fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
    ax.plot(
        video_recall,
        video_precision,
        label=f"Average Precision = {video_ap:.4f}",
    )
    ax.set_title(
        "Test Precision-Recall Curve — Video Level"
    )
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_figure(
        fig,
        "test_precision_recall_curve_video_level.png",
    )

print(json.dumps(video_metrics, indent=2))

{
  "accuracy": 0.6521739130434783,
  "balanced_accuracy": 0.6075373619233269,
  "precision": 0.6542056074766355,
  "recall": 0.8641975308641975,
  "f1": 0.7446808510638298,
  "roc_auc": 0.6826943902967294,
  "average_precision": 0.7549471855705238,
  "evaluation_level": "video",
  "mode": "full",
  "selected_threshold": 0.33,
  "video_count": 138,
  "frame_count": 196,
  "aggregation": "mean_probability"
}


In [21]:
# ============================================================
# 19. OPTIONAL ABLATION COMPARISON
# ============================================================
ablation_path = (
    DIRS["metrics"] / "ablation_results.csv"
)
ablation = pd.read_csv(ablation_path)

if len(ablation) > 1:
    fig, ax = plt.subplots(figsize=(12, 7), dpi=150)
    positions = np.arange(len(ablation))
    width = 0.25

    ax.bar(
        positions - width,
        ablation["accuracy"],
        width,
        label="Accuracy",
    )
    ax.bar(
        positions,
        ablation["f1"],
        width,
        label="F1 Score",
    )
    ax.bar(
        positions + width,
        ablation["roc_auc"],
        width,
        label="ROC AUC",
    )

    ax.set_xticks(
        positions,
        ablation["mode"],
        rotation=20,
        ha="right",
    )
    ax.set_title(
        "Ablation Study Performance Comparison"
    )
    ax.set_xlabel("Feature Configuration")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    save_figure(
        fig,
        "ablation_performance_comparison.png",
    )
else:
    print(
        "Tek model çalıştırıldı. Tüm kombinasyonlar için "
        "CONFIG['run_all_ablations'] = True yapabilirsin."
    )

Tek model çalıştırıldı. Tüm kombinasyonlar için CONFIG['run_all_ablations'] = True yapabilirsin.


In [22]:
# ============================================================
# 20. FINAL QUALITY GATES, RUN SUMMARY AND OUTPUT MANIFEST
# ============================================================
required_outputs = [
    DIRS["checkpoints"] / "last.ckpt",
    DIRS["checkpoints"] / "best.ckpt",
    DIRS["logs"] / "pipeline.log",
    DIRS["logs"] / "training_history.csv",
    DIRS["metrics"] / "data_accounting.json",
    DIRS["metrics"] / "epoch_metrics.csv",
    DIRS["metrics"] / "test_metrics_frame_level.json",
    DIRS["metrics"] / "test_metrics_video_level.json",
    DIRS["metrics"] / "test_metrics_summary.csv",
    DIRS["predictions"]
    / "test_predictions_frame_level.csv",
    DIRS["predictions"]
    / "test_predictions_video_level.csv",
    DIRS["figures"] / "dataset_distribution.png",
    DIRS["artifacts"] / "metadata_used.csv",
    DIRS["artifacts"] / "texture_scaler.joblib",
    DIRS["artifacts"] / "feature_dimensions.json",
    RUN_DIR / "config_resolved.yaml",
    RUN_DIR / "requirements_lock.txt",
    RUN_DIR / "environment.json",
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]
if missing_outputs:
    raise AssertionError(
        "Eksik deney çıktıları:\n"
        + "\n".join(missing_outputs)
    )

checkpoint = torch.load(
    DIRS["checkpoints"] / "best.ckpt",
    map_location="cpu",
    weights_only=False,
)
for required_key in [
    "model_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
]:
    if required_key not in checkpoint:
        raise AssertionError(
            f"Best checkpoint içinde {required_key} yok."
        )

epoch_metrics = pd.read_csv(
    DIRS["metrics"] / "epoch_metrics.csv"
)
numeric_columns = epoch_metrics.select_dtypes(
    include=[np.number]
).columns
if not np.isfinite(
    epoch_metrics[numeric_columns].to_numpy()
).all():
    raise FloatingPointError(
        "Epoch metrics içinde NaN/Inf bulundu."
    )

frame_predictions = pd.read_csv(
    DIRS["predictions"]
    / "test_predictions_frame_level.csv"
)
if frame_predictions.empty:
    raise AssertionError(
        "Frame prediction dosyası boş."
    )
if not frame_predictions["sample_id"].is_unique:
    raise AssertionError(
        "Test prediction sample_id değerleri benzersiz değil."
    )

# Figure audit
figure_rows = []
for figure_path in sorted(
    DIRS["figures"].glob("*.png")
):
    with Image.open(figure_path) as image:
        width, height = image.size
    figure_rows.append({
        "file": figure_path.name,
        "width_px": int(width),
        "height_px": int(height),
        "short_edge_px": int(min(width, height)),
        "minimum_600_px_passed": bool(
            min(width, height) >= 600
        ),
    })

figure_audit = pd.DataFrame(figure_rows)
atomic_write_csv(
    figure_audit,
    DIRS["metrics"]
    / "figure_quality_audit.csv",
)
if (
    not figure_audit.empty
    and not figure_audit[
        "minimum_600_px_passed"
    ].all()
):
    raise AssertionError(
        "600 px altı figür bulundu."
    )

final_quality_gates = {
    "required_outputs": "PASSED",
    "checkpoint_integrity": "PASSED",
    "numeric_metrics": "PASSED",
    "prediction_uniqueness": "PASSED",
    "figure_resolution": "PASSED",
    "metadata_video_leakage": "PASSED",
}
atomic_write_json(
    final_quality_gates,
    DIRS["metrics"] / "quality_gates.json",
)

frame_metrics = json.loads(
    (
        DIRS["metrics"]
        / "test_metrics_frame_level.json"
    ).read_text(encoding="utf-8")
)
video_metrics = json.loads(
    (
        DIRS["metrics"]
        / "test_metrics_video_level.json"
    ).read_text(encoding="utf-8")
)
data_accounting = json.loads(
    (
        DIRS["metrics"]
        / "data_accounting.json"
    ).read_text(encoding="utf-8")
)

RUN_COMPLETED_AT = datetime.now(timezone.utc)
run_summary = {
    "run_id": RUN_ID,
    "region": CONFIG["region"],
    "model_name": CONFIG["model_name"],
    "active_mode": CONFIG["active_mode"],
    "feature_extractors": [
        "LBP",
        "GLCM",
        "Gabor",
        "Wavelet",
    ],
    "feature_counts": FEATURE_COUNTS,
    "total_handcrafted_feature_dimension": int(
        len(FEATURE_NAMES)
    ),
    "frame_level_metrics": frame_metrics,
    "video_level_metrics": video_metrics,
    "data_accounting": data_accounting,
    "quality_gates": final_quality_gates,
    "started_at_utc": RUN_STARTED_AT.isoformat(),
    "completed_at_utc": RUN_COMPLETED_AT.isoformat(),
    "duration_seconds": round(
        time.monotonic() - RUN_START_MONOTONIC,
        3,
    ),
    "run_directory": str(RUN_DIR),
}
atomic_write_json(
    run_summary,
    RUN_DIR / "run_summary.json",
)

# Manifest is written last. It intentionally excludes itself,
# because a file cannot contain its own stable hash.
manifest_rows = []
for output_path in sorted(
    path
    for path in RUN_DIR.rglob("*")
    if path.is_file()
    and path.name != "output_manifest.csv"
):
    stat = output_path.stat()
    manifest_rows.append({
        "relative_path": str(
            output_path.relative_to(RUN_DIR)
        ),
        "size_bytes": int(stat.st_size),
        "sha256": sha256_file(output_path),
        "modified_at_utc": datetime.fromtimestamp(
            stat.st_mtime,
            tz=timezone.utc,
        ).isoformat(),
    })

atomic_write_csv(
    pd.DataFrame(manifest_rows),
    RUN_DIR / "output_manifest.csv",
)

log("Experiment completed successfully.")
log(f"Output directory: {RUN_DIR}")
print(json.dumps(run_summary, indent=2, ensure_ascii=False))

2026-08-06 22:38:35,414 | INFO | Experiment completed successfully.


INFO:eyebrow_experiment:Experiment completed successfully.


2026-08-06 22:38:35,416 | INFO | Output directory: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260806_2124_eyebrow_swinv2_lbp_glcm_gabor_wavelet_seed42


INFO:eyebrow_experiment:Output directory: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260806_2124_eyebrow_swinv2_lbp_glcm_gabor_wavelet_seed42


{
  "run_id": "20260806_2124_eyebrow_swinv2_lbp_glcm_gabor_wavelet_seed42",
  "region": "eyebrow",
  "model_name": "swinv2_t_lbp_glcm_gabor_wavelet_fusion",
  "active_mode": "full",
  "feature_extractors": [
    "LBP",
    "GLCM",
    "Gabor",
    "Wavelet"
  ],
  "feature_counts": {
    "lbp": 54,
    "glcm": 72,
    "gabor": 72,
    "wavelet": 42
  },
  "total_handcrafted_feature_dimension": 240,
  "frame_level_metrics": {
    "accuracy": 0.6020408163265306,
    "balanced_accuracy": 0.6091714434601354,
    "precision": 0.5594405594405595,
    "recall": 0.8421052631578947,
    "f1": 0.6722689075630253,
    "roc_auc": 0.6687858259510162,
    "average_precision": 0.6707913954129812,
    "evaluation_level": "frame",
    "mode": "full",
    "best_epoch": 5,
    "selection_metric": "val_roc_auc",
    "best_validation_auc": 0.727955333076627,
    "selected_threshold": 0.33,
    "texture_dimension": 240,
    "feature_groups": [
      "lbp",
      "glcm",
      "gabor",
      "wavelet"
    ],